After many iterations, the definitive approach for framing will use the **Polars** library (an efficient alternative to Pandas) together with **imageio.v3**. The latter allows us to iterate over video frames and save them directly as bytes.

We chose to store frames as bytes because our previous library (**MoviePy**) returned NumPy arrays, which are more difficult to convert into PNG images for sending to the LLM via an OpenAI-compatible API. This decision was partly inspired by earlier work where we sent images of charts to evaluate the AI’s ability to interpret them.

The **Parquet** format emerged after many hours of experimentation and inspiration from Hugging Face. Other formats proved highly inefficient for compressing frames, resulting in files that were too large on disk.

In [7]:
import os
import imageio.v3 as iio
import polars as pl
import pandas as pd 

Loop for generating the parquet file

In [10]:
rows = [] #Empty list for storing each frame
source = os.getenv("VIDEO_PATH", "data/match.mp4")
#metadata = iio.immeta(source) 
#source_fps = metadata.get("fps") # Code to see the fps of the video (25 in this case)

N1= 79500#start game 
N2 = 151500 # end of second time
#fps range of the frames that we want to store
for i in range(N1, N2, 12): 
    frame = iio.imread(source,index=i,plugin="pyav",)
    png_encoded = iio.imwrite("<bytes>", frame, extension=".png")
    rows.append({"frame_col":png_encoded})
    #This is a loop that goes the indicated frames, step of 12 to generate around 2 frames per second, Storing optimization and VL caplabe
df = pl.DataFrame(rows)
df.write_parquet(file= "data/second_half.parquet", compression= "zstd", compression_level=22)

In [11]:
data = pl.read_parquet("data/second_half.parquet") #polar is mandatory